# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 5/5 [06:34<00:00, 78.97s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

"Title: Kitchens of India Paste of Butter Chicken 6-Pack for $11 + free shipping w/ $35\nDetails: It's $3 off, and a buck under the last time we listed it. Shipping adds $6.99, or orders of $35 or more ship for free.\xa0 Buy Now at Walmart\nFeatures: \nURL: https://www.dealnews.com/Kitchens-of-India-Paste-of-Butter-Chicken-6-Pack-for-11-free-shipping-w-35/21752042.html?iref=rss-c196"

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Samsung Galaxy Watch Ultra 47mm LTE Smartwatch (2025) Pre-Order: $50 off + up to $250 off w/ trade-in + free shipping
Details: It's now $500 with no trade required, $50 under our mention from yesterday. Plus, if you have a trade, you'll save up to another $250 off. The watch is due to release on July 25.  Shop Now at Samsung
Features: 
URL: https://www.dealnews.com/

In [ ]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed #Parse to an instance of DealSelection
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [17]:
result.deals[1]

Deal(product_description='The Jackery 5000 Plus is a powerful energy solution for homes, featuring a massive 5,040Wh lithium iron phosphate (LiFePO4) battery, capable of expanding to 60kWh for your power needs. It includes smart app control and a 7,200W dual voltage output, allowing for seamless integration into your power supply strategy. Perfect for off-grid situations or as a safety net during outages, it ensures you have continuous access to energy.', price=2799.0, url='https://www.dealnews.com/products/Jackery/Jackery-5000-Plus-5-040-Wh-Power-Station/484764.html?iref=rss-c142')

In [3]:
from agents.scanner_agent import ScannerAgent

In [4]:
agent = ScannerAgent()
result = agent.scan()

In [9]:
result.deals[0].json()

C:\Users\ssre_\AppData\Local\Temp\ipykernel_7088\543935192.py:1: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  result.deals[0].json()


'{"product_description":"The Samsung Galaxy Watch Ultra is a premium smartwatch featuring a 47mm display with LTE connectivity, designed for fitness enthusiasts and tech-savvy users alike. It boasts a robust battery life, advanced health monitoring capabilities, and integrates seamlessly with other Samsung devices. With its durable design and cutting-edge technology, this smartwatch is ideal for tracking workouts and staying connected on the go.","price":500.0,"url":"https://www.dealnews.com/Samsung-Galaxy-Watch-Ultra-47-mm-LTE-Smartwatch-2025-Pre-Order-50-off-up-to-250-off-w-trade-in-free-shipping/21752168.html?iref=rss-c142"}'